In [ ]:
# ==============================================================================
# ML PRACTICAL 1: UBER FARE PREDICTION
# Objective: Predict Uber fares using Linear Regression and Random Forest Regressor
# ==============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

# 2. Load Dataset
df = pd.read_csv("uber.csv")
print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

# 3. Data Cleaning & Feature Engineering
# Drop missing values and unneeded columns
df = df.dropna().drop(columns=['Unnamed: 0', 'key'], errors='ignore')

# Extract pickup hour
df['time'] = pd.to_datetime(df['pickup_datetime']).dt.hour
df = df.drop(columns=['pickup_datetime'])

# Calculate Euclidean distance between pickup and dropoff coordinates
df['distance'] = np.sqrt(
    (df['dropoff_latitude'] - df['pickup_latitude'])**2 + 
    (df['dropoff_longitude'] - df['pickup_longitude'])**2
)

print("\nEngineered Data Preview:")
print(df.head())

# 4. Outlier Removal (IQR)
# Filter invalid passenger counts
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] < 150)]

# Filter coordinate outliers using Interquartile Range (IQR)
coord_cols = ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude']
for col in coord_cols:
    q1, q3 = np.percentile(df[col], [25, 75])
    iqr = q3 - q1
    df = df[(df[col] >= q1 - 1.5 * iqr) & (df[col] <= q3 + 1.5 * iqr)]

print(f"\nCleaned Dataset Shape: {df.shape}")

# 5. Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight='bold')
plt.show()

# 6. Train-Test Split & Feature Scaling
X = df.drop(columns=['fare_amount'])
y = df['fare_amount']

# 80:20 Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Model Training
# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
print("\nModels trained successfully!")

# 8. Evaluation & Comparison Table
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regressor'],
    'R2 Score': [r2_score(y_test, y_pred_lr), r2_score(y_test, y_pred_rf)],
    'RMSE': [root_mean_squared_error(y_test, y_pred_lr), root_mean_squared_error(y_test, y_pred_rf)]
})

print("\n--- Model Evaluation Summary ---")
print(results)

# 9. Comparison Plots
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Plot R2 Score
sns.barplot(data=results, x='Model', y='R2 Score', ax=axes[0], palette='Blues_d')
axes[0].set_title('R² Score Comparison (Higher is Better)', fontweight='bold')
axes[0].set_ylim(0, 1)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.4f}", 
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', color='white', fontweight='bold', fontsize=11)

# Plot RMSE
sns.barplot(data=results, x='Model', y='RMSE', ax=axes[1], palette='Reds_d')
axes[1].set_title('RMSE Comparison (Lower is Better)', fontweight='bold')
axes[1].set_ylim(0, max(results['RMSE']) * 1.2)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.4f}", 
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', color='white', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

# ==============================================================================
# VIVA QUICK REFERENCE:
# 1. R² Score: Proportion of variance in fare explained by features (closer to 1 is better).
# 2. RMSE: Root Mean Squared Error, penalizes larger prediction errors (lower is better).
# 3. Why Random Forest outperforms Linear Regression: Random Forest captures complex
#    non-linear relationships and feature interactions (e.g., distance & time of day).
# ==============================================================================